In [1]:
import psycopg2

In [160]:
import sys
import re
import numpy as np
import pandas as pd
import hashlib
import sqlite3
import xml.etree.ElementTree as ET


In [165]:
from datasets import concatenate_datasets,load_from_disk,Dataset
import json

In [148]:
def get_sqlite_data(tbl):
    db_file = f"/media/sunveil/Data/header_detection/poddubnyy/postgraduate/squall/tables/db/{tbl}.db"
    conn = sqlite3.connect(db_file)
    df = pd.read_sql_query("SELECT * FROM w", conn)
    del df['id']
    return df

In [3]:
sys.path.append('../TapexGraph')

In [4]:
from add_utils import deserializ_tapex_linear_table,escape_special_characters

In [5]:
def read_questions(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            yield line.strip()

In [93]:
questions_gen = read_questions('/media/sunveil/Data/header_detection/poddubnyy/postgraduate/OmniGraph/TapexGraph/tapex_pretrain/train.src')
question = list(questions_gen)

In [122]:
dataset = Dataset.from_dict({
        'question': question})
dataset = dataset.filter(lambda x: True if 'from' in x['question'][:20].lower() else False)

Filter:   0%|          | 0/4966214 [00:00<?, ? examples/s]

In [124]:
len(dataset['question'])

7960

In [125]:
dataset['question']

["select from where party = 'se' col : from | to | name | party | position row 1 : 1850 | 1862 | m. borgarelli d'ison | none | colonel of infantry e.r row 2 : 1862 | 1869 | jules alfred le tourneur du coudray | none | first secretary for the minister of finance row 3 : 1869 | 1903 | stanislas le tourneur d'ison | none | attached to the taxation administration of france row 4 : none | none | yves le tourneur d'ison | none | doctor of juridical science row 5 : 1966 | 1989 | maurice lecoq | none | market gardener row 6 : 1989 | 2008 | michel deuzet | se | farmer row 7 : 2008 | 2014 | patrice martin | se | bank officer",
 "select from where party = 'se' col : from | to | name | party | position row 1 : 1850 | 1862 | m. borgarelli d'ison | none | colonel of infantry e.r row 2 : 1862 | 1869 | jules alfred le tourneur du coudray | none | first secretary for the minister of finance row 3 : 1869 | 1903 | stanislas le tourneur d'ison | none | attached to the taxation administration of france row

In [72]:
def get_sql_and_table(example):
    pattern = ' col : '
    try:
        sql,table = example.split(pattern)
    except Exception as e:
        #print(e)
        return "None", "None"
    sql  = sql.strip()
    df = deserializ_tapex_linear_table(" col : "+table)
    
    new_column_names = [f'"{"_".join(head.split(" "))}"' for head in df.columns]
    #print(new_column_names)
    for head in sorted(set(df.columns),key=len, reverse=True):
        sql = sql.replace(head,f'{"_".join(head.split(" "))}')
    #print(sql)
    for head in set(df.columns):  
        #print(escape_special_characters("_".join(head.split(" "))))
        sql = re.sub(f' {escape_special_characters("_".join(head.split(" ")))} '
                     ,f" '{'_'.join(head.split(' '))}' ",sql) 
        #sql = sql.replace(head,f'"{"_".join(head.split(" "))}"') 
        #print(sql)
    df.columns = new_column_names
    del new_column_names
    df['agg'] = np.zeros(df.shape[0])
    return sql,df

In [73]:
def get_psql_type(type_):
    if type_ == int:
        return 'INTEGER'

    elif type_ == float:
        
        return 'REAL'

    else:
        return 'TEXT'

In [131]:
def covichki(p):
    return f"'{p}'"
def join_params(params):
    #return f"({','.join(params.apply(lambda x:covichki(x) if type(x) == str else str(x)).values)})"
    return f"({','.join(params.apply(lambda x:str(x)).values)})"

In [169]:
def get_query_execution_plan(table,sql):
    host = '192.168.19.148'
    database = 'tapex'
    user = 'postgres'
    password = '0000'
    port = 5432
    answer = None
    try:
        tab_name = 'w'
        explain_sql = 'EXPLAIN (ANALYZE,FORMAT XML)' + sql
        create_table_query = f'CREATE TABLE IF NOT EXISTS {tab_name} (id SERIAL PRIMARY KEY,'\
                            f'{",".join([f"{key} {get_psql_type(table.dtypes[key])}" for key in table.columns])});'
        #insert_data_query = f'INSERT INTO {tab_name} ({",".join([key for key in data.columns if key !="id"])}) VALUES '\
                            #f'{",".join(["%s" for _,x in data.iterrows()])};'
        insert_data_query = f'INSERT INTO {tab_name} ({",".join([key for key in table.columns])}) VALUES '\
                            f'({",".join(["%s" for key in table.columns ])});'
        drop_table_query = f'DROP TABLE IF EXISTS {tab_name};'
        # Подключение к базе данных
        connection = psycopg2.connect(
            host=host,
            database=database,
            user=user,
            password=password,
            port=port
        )
        cursor = connection.cursor()
        # Получение информации о версии postgres
        
        
        cursor.execute(create_table_query)
        
        
    
        
    
        # Используем executemany для вставки множества строк
    
        cursor.executemany(insert_data_query, [x.to_list() for _,x in table.iterrows()])  
        cursor.execute(explain_sql)
        answer = cursor.fetchall()
        cursor.execute(drop_table_query)
        connection.commit()
        #version = cursor.fetchone()[0]
        #print(f"Версия PostgreSQL: {version}")
    except Exception as e:
        print("Ошибка при работе с БД:", e)
    finally:
        if cursor:
            cursor.close()  # Закрытие курсора
        if connection:
            connection.close()  # Закрытие соединения
        return answer    

In [167]:
with open('/media/sunveil/Data/header_detection/poddubnyy/postgraduate/squall/data/squall.json','r') as inf:
    squall = json.load(inf)
def get_sqall_execution_plan(squall_example):
    table = get_sqlite_data(squall_example['tbl'])
    sql = ' '.join([s[1] for s in squall_example['sql']])
    answer = get_query_execution_plan(table,sql)
    return answer[0][0] if answer != None else answer

In [181]:
x = get_sqall_execution_plan(squall[0])

Ошибка при работе с БД: more than one row returned by a subquery used as an expression



In [173]:
host = '192.168.19.148'
database = 'tapex'
user = 'postgres'
password = '0000'
port = 5432
try:
    
    #sql,data = get_sql_and_table(test)
    #tab_name = hashlib.md5(test.encode('utf-8')).hexdigest()
    #create_table_query = f'CREATE TABLE IF NOT EXISTS {tab_name} (id SERIAL PRIMARY KEY,'\
     #                   f'{",".join([f"{key} {get_psql_type(data.dtypes[key])}" for key in data.columns])});'
    #insert_data_query = f'INSERT INTO {tab_name} ({",".join(data.columns)}) VALUES '\
      #                  f'{",".join([join_params(x) for _,x in data.iterrows()])};'
    data = get_sqlite_data(pp[0]['tbl'])
    tab_name = 'w'
    sql = ' '.join([s[1] for s in pp[0]['sql']])
    explain_sql = 'EXPLAIN (ANALYZE,FORMAT XML)' + sql
    create_table_query = f'CREATE TABLE IF NOT EXISTS {tab_name} (id SERIAL PRIMARY KEY,'\
                        f'{",".join([f"{key} {get_psql_type(data.dtypes[key])}" for key in data.columns])});'
    #insert_data_query = f'INSERT INTO {tab_name} ({",".join([key for key in data.columns if key !="id"])}) VALUES '\
                        #f'{",".join(["%s" for _,x in data.iterrows()])};'
    insert_data_query = f'INSERT INTO {tab_name} ({",".join([key for key in data.columns])}) VALUES '\
                        f'({",".join(["%s" for key in data.columns ])});'
    drop_table_query = f'DROP TABLE IF EXISTS {tab_name};'
    # Подключение к базе данных
    connection = psycopg2.connect(
        host=host,
        database=database,
        user=user,
        password=password,
        port=port
    )
    cursor = connection.cursor()
    # Получение информации о версии postgres
    
    
    cursor.execute(create_table_query)
    
    

    

    # Используем executemany для вставки множества строк

    cursor.executemany(insert_data_query, [x.to_list() for _,x in data.iterrows()])  
    #cursor.execute(explain_sql)
    #answer = cursor.fetchall()
    #cursor.execute(drop_table_query)
    connection.commit()
    #version = cursor.fetchone()[0]
    #print(f"Версия PostgreSQL: {version}")
except Exception as e:
    print("Ошибка при работе с БД:", e)
finally:
    if cursor:
        cursor.close()  # Закрытие курсора
    if connection:
        connection.close()  # Закрытие соединения
    

In [20]:
def identify_type(input_string):
    try:
        # Попробуем преобразовать строку в целое число
        value = int(input_string)
        return value
    except ValueError:
        try:
            # Попробуем преобразовать строку в дробное число
            value = float(input_string)
            return value
        except ValueError:
            # Если ничего не сработало, это текст
            return input_string
    except TypeError:
        return input_string